# Estudo de caso 2.3 - Os países pobres crescem mais rápido que os países ricos?


Configuração do *notebook*:

Sincronize sua conta do Google. Para isso, siga o link que aparece na saída da seguinte célula uma vez executada. Copie o código que aparece na tela e insira-o na saída da célula. Assim que visualizar a mensagem: `Google Drive sincronizado com sucesso!` poderá continuar executando o restante das células.

In [ ]:
!pip install rpy2==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.7/201.7 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rpy2: filename=rpy2-3.5.1-cp310-cp310-linux_x86_64.whl size=318083 sha256=f27db788af6e48f9e0ebc24e4ff30d0a577dc2c7f32c4085ac5a260dadb1812e
  Stored in directory: /root/.cache/pip/wheels/73/a6/ff/4e75dd1ce1cfa2b9a670cbccf6a1e41c553199e9b25f05d953
Successfully built rpy2
  Attempting uninstall: rpy2
    Found existing installation: rpy2 3.5.5
    Uninstalling rpy2-3.5.5:
      Successfully uninstalled rpy2-3.5.5


In [ ]:
from google.colab import auth
auth.authenticate_user()

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
#data_drop = drive.CreateFile({'id':'1f4F3EcARvrQeoNqNfWYNcWhjV5dGnulc'})
data_drop = drive.CreateFile({'id':'1wAZ5JZuq2_zKqzDQlIuOxUSTfk9N_tq5'})
data_drop.GetContentFile('growth.Rdata')

print('Google Drive sincronizado com sucesso!')

Google Drive sincronizado com sucesso!


In [ ]:
%load_ext rpy2.ipython


Instalar e importar bibliotecas: (ignorar resultados a não ser que não apareça a frase: `Bibliotecas instaladas com sucesso!`)

In [ ]:
%%R
install.packages("hdm")

cat('\nBibliotecas instaladas com sucesso!')

(as ‘lib’ is unspecified)

















































	‘/tmp/Rtmp3Z6EJx/downloaded_packages’




Bibliotecas instaladas com sucesso!

In [ ]:
%%R
# Importar bibliotecas
library(hdm)

cat('\nBibliotecas importadas com sucesso!')


Bibliotecas importadas com sucesso!

## Dados


In [ ]:
%%R
# Carregar o banco de dados
load(file="growth.Rdata")

# Ver as variáveis do banco de dados
class(growth)
str(growth)

# Mostrar dimensões do banco de dados
dims  <- dim(growth)
cat('\nDimensões do banco de dados:',toString(dims),'\n',fill = TRUE)

'data.frame':	90 obs. of  63 variables:
 $ Outcome  : num  -0.0243 0.1005 0.0671 0.0641 0.0279 ...
 $ intercept: int  1 1 1 1 1 1 1 1 1 1 ...
 $ gdpsh465 : num  6.59 6.83 8.9 7.57 7.16 ...
 $ bmp1l    : num  0.284 0.614 0 0.2 0.174 ...
 $ freeop   : num  0.153 0.314 0.204 0.249 0.299 ...
 $ freetar  : num  0.04389 0.06183 0.00919 0.03627 0.03737 ...
 $ h65      : num  0.007 0.019 0.26 0.061 0.017 0.023 0.039 0.024 0.402 0.145 ...
 $ hm65     : num  0.013 0.032 0.325 0.07 0.027 0.038 0.063 0.035 0.488 0.173 ...
 $ hf65     : num  0.001 0.007 0.201 0.051 0.007 0.006 0.014 0.013 0.314 0.114 ...
 $ p65      : num  0.29 0.91 1 1 0.82 0.5 0.92 0.69 1 1 ...
 $ pm65     : num  0.37 1 1 1 0.85 0.55 0.94 0.69 1 1 ...
 $ pf65     : num  0.21 0.65 1 1 0.81 0.5 0.92 0.69 1 1 ...
 $ s65      : num  0.04 0.16 0.56 0.24 0.17 0.08 0.17 0.14 0.9 0.28 ...
 $ sm65     : num  0.06 0.23 0.62 0.22 0.15 0.1 0.21 0.14 0.9 0.26 ...
 $ sf65     : num  0.02 0.09 0.51 0.31 0.13 0.07 0.12 0.13 0.9 0.4 ...
 $ fert65

In [ ]:
%%R
# Obter os nomes das variáveis
varnames= colnames(growth)
# Extrair os nomes dos controles e do regressor objetivo de varnames
xnames     <- varnames[-c(1,2,3)]     # nomes das variáveis de X
dandxnames <- varnames[-c(1,2)]       # nomes das variáveis de X e D

## Metodologia

In [ ]:
%%R
# criar fórmulas "colando" nomes (para salvar os nomes a serem mostrados)
fmla      <-  as.formula(paste("Outcome ~ ", paste(dandxnames, collapse= "+")))
full.fit  <-  lm(fmla, data=growth)
fmla.y    <-  as.formula(paste("Outcome ~ ", paste(xnames, collapse= "+")))
fmla.d    <-  as.formula(paste("gdpsh465~ ", paste(xnames, collapse= "+")))


# d particionado e y particionado por regressão linear
rY       <- rlasso(fmla.y, data =growth)$res
rD       <- rlasso(fmla.d, data =growth)$res


# regressão linear de Y particionado no D particionado
partial.fit.lasso <- lm(rY~rD-1)

# criar tabela para coletar os resultados
table      <- matrix(0, 2, 4)
table[1,1:2]  <- summary(full.fit)$coef["gdpsh465",1:2]
table[2,1:2]  <- summary(partial.fit.lasso)$coef[1,1:2]
table[1,3:4]  <- confint(full.fit, level=0.95, 'gdpsh465')[1:2]
table[2,3:4]  <- confint(partial.fit.lasso, level=0.95, 'rD')[1:2]

# atribuir nomes de colunas e filas
colnames(table) <- c("Estimativa (beta)", "Erro padrão","2,5%","97,5%")
rownames(table) <- c("Mínimos quadrados ord. (OLS)", "Particionamento com Lasso")

In [ ]:
%%R
print(fmla.d)

gdpsh465 ~ bmp1l + freeop + freetar + h65 + hm65 + hf65 + p65 + 
    pm65 + pf65 + s65 + sm65 + sf65 + fert65 + mort65 + lifee065 + 
    gpop1 + fert1 + mort1 + invsh41 + geetot1 + geerec1 + gde1 + 
    govwb1 + govsh41 + gvxdxe41 + high65 + highm65 + highf65 + 
    highc65 + highcm65 + highcf65 + human65 + humanm65 + humanf65 + 
    hyr65 + hyrm65 + hyrf65 + no65 + nom65 + nof65 + pinstab1 + 
    pop65 + worker65 + pop1565 + pop6565 + sec65 + secm65 + secf65 + 
    secc65 + seccm65 + seccf65 + syr65 + syrm65 + syrf65 + teapri65 + 
    teasec65 + ex1 + im1 + xr65 + tot1


## Resultados

In [ ]:
%%R
# Mostrar resultados
print(table, digits=2)

                             Estimativa (beta) Erro padrão   2,5%  97,5%
Mínimos quadrados ord. (OLS)           -0.0094       0.030 -0.071  0.052
Particionamento com Lasso              -0.0498       0.014 -0.077 -0.022
